<a href="https://colab.research.google.com/github/sanaisrail/code-switching-codesaviours-si26-sana/blob/main/SI26_Week7_Sana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [34]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset (4).csv")

print(df.head())
print("\nColumns:")
print(df.columns.tolist())

print("\nShape:")
print(df.shape)

                                            sentence    word label
0  Main thori der mein online aa jaungi, just wai...    Main   ENG
1  Main thori der mein online aa jaungi, just wai...   thori   URD
2  Main thori der mein online aa jaungi, just wai...     der   URD
3  Main thori der mein online aa jaungi, just wai...    mein   ENG
4  Main thori der mein online aa jaungi, just wai...  online   URD

Columns:
['sentence', 'word', 'label']

Shape:
(2166, 3)


In [35]:
print("Labels:")
print(df["label"].value_counts())

print("\nUnique labels:")
print(df["label"].unique())

Labels:
label
URD    1084
ENG    1042
MIX      40
Name: count, dtype: int64

Unique labels:
['ENG' 'URD' 'MIX']


In [36]:
# Check how many sentences contain both URD and ENG words

sentence_labels = df.groupby("sentence")["label"].unique()

mixed_sentences = sentence_labels[
    sentence_labels.apply(lambda x: len(x) > 1)
]

print("Total sentences:", len(sentence_labels))
print("Mixed-language sentences:", len(mixed_sentences))

print("\nExample mixed sentences:")
for sentence in mixed_sentences.index[:5]:
    print(sentence)

Total sentences: 200
Mixed-language sentences: 200

Example mixed sentences:
Aaj bohot zyada assignments hain, I don't know how I will finish them.
Aaj class ke baad we can discuss the project, agar tum free ho.
Aaj class mein new topic start hua, and it was actually interesting.
Aaj class mein students bohot active thay, everyone participated in the discussion.
Aaj hum friends ke saath shopping karne ja rahe hain, do you want to join?


In [37]:
sentences = (
    df.groupby("sentence")
      .apply(
          lambda x: {
              "words": x["word"].tolist(),
              "labels": x["label"].tolist()
          }
      )
      .tolist()
)

print("Total grouped sentences:", len(sentences))

print("\nFirst sentence:")
print(sentences[0])

Total grouped sentences: 200

First sentence:
{'words': ['Aaj', 'bohot', 'zyada', 'assignments', 'hain,', 'I', "don't", 'know', 'how', 'I', 'will', 'finish', 'them.'], 'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']}


/tmp/ipykernel_2128/1165086530.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [38]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 160
Testing sentences: 40


In [76]:
# Label mapping
label2id = {
    "URD": 0,
    "ENG": 1,
    "MIX": 2
}

id2label = {
    0: "URD",
    1: "ENG",
    2: "MIX"
}


def tokenize_and_align_labels(examples):

    # Words ko tokens mein convert karo
    tokenized = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    # Har sentence ke labels process karo
    for i, label in enumerate(examples["labels"]):

        # Har token kis original word se belong karta hai
        word_ids = tokenized.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:

            # Special token
            if word_id is None:
                label_ids.append(-100)

            # Word ka first token
            elif word_id != previous_word_id:
                label_ids.append(
                    label2id[label[word_id]]
                )

            # Same word ka additional sub-token
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels

    return tokenized

print("Tokenization function ready!")
print("Labels:", label2id)

Tokenization function ready!
Labels: {'URD': 0, 'ENG': 1, 'MIX': 2}


In [77]:
from sklearn.model_selection import train_test_split

# Fresh dataset se sentences group karo
sentences = df.groupby("sentence").apply(
    lambda x: {
        "words": x["word"].tolist(),
        "labels": x["label"].tolist()
    }
).tolist()

# 80% training, 20% testing
train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 160
Testing sentences: 40


/tmp/ipykernel_2128/1244648105.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby("sentence").apply(


In [78]:
from datasets import Dataset

def to_hf_dataset(data):
    return Dataset.from_dict({
        "words": [item["words"] for item in data],
        "labels": [item["labels"] for item in data]
    })

train_ds = to_hf_dataset(train_data)
test_ds = to_hf_dataset(test_data)

print("Training dataset:")
print(train_ds)

print("\nTesting dataset:")
print(test_ds)

Training dataset:
Dataset({
    features: ['words', 'labels'],
    num_rows: 160
})

Testing dataset:
Dataset({
    features: ['words', 'labels'],
    num_rows: 40
})


In [46]:
import torch

from transformers import (
    TrainingArguments,
    DataCollatorForTokenClassification
)

# Data ko batches mein properly arrange karne ke liye
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

# Training settings
training_args = TrainingArguments(
    output_dir="./results",

    # 5 epochs
    num_train_epochs=5,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Har epoch ke baad evaluation
    eval_strategy="epoch",

    # Har epoch ke baad model save
    save_strategy="epoch",

    # Har 10 steps ke baad logging
    logging_steps=10,

    # Best model ko training ke end par load karna
    load_best_model_at_end=True,

    # External reporting band
    report_to="none",

    # GPU available ho to mixed precision
    fp16=torch.cuda.is_available()
)

print("Training settings ready!")
print("GPU available:", torch.cuda.is_available())

Training settings ready!
GPU available: True


In [47]:
# Training aur testing dataset ko tokenize karo

tokenized_train = train_ds.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_test = test_ds.map(
    tokenize_and_align_labels,
    batched=True
)

print("Training tokenization complete!")
print("Testing tokenization complete!")

print("Columns:", tokenized_train.column_names)

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Training tokenization complete!
Testing tokenization complete!
Columns: ['words', 'labels', 'input_ids', 'attention_mask']


In [66]:
from transformers import AutoModelForTokenClassification

model_name = "xlm-roberta-base"

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={
        0: "URD",
        1: "ENG",
        2: "MIX"
    },
    label2id={
        "URD": 0,
        "ENG": 1,
        "MIX": 2
    }
)

print("Fresh model loaded!")
print(model.config.id2label)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh model loaded!
{0: 'URD', 1: 'ENG', 2: 'MIX'}


In [68]:
def predict_labels(sentence):
    words = sentence.split()

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    predictions = torch.argmax(outputs.logits, dim=-1)[0]

    word_ids = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True
    ).word_ids()

    results = []
    previous_word_id = None

    for token_idx, word_id in enumerate(word_ids):

        if word_id is None:
            continue

        if word_id != previous_word_id:
            label_id = predictions[token_idx].item()
            label = model.config.id2label[label_id]

            results.append(
                (words[word_id], label)
            )

        previous_word_id = word_id

    return results


print(predict_labels("Maine msging ki"))

[('Maine', 'MIX'), ('msging', 'MIX'), ('ki', 'MIX')]


In [69]:
tokenized_train = train_ds.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_test = test_ds.map(
    tokenize_and_align_labels,
    batched=True
)

print("DONE")
print(tokenized_train.column_names)

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

DONE
['words', 'labels', 'input_ids', 'attention_mask']


In [70]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

print("Trainer ready!")

Trainer ready!


In [71]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.084307,0.553332
2,0.486886,0.389499
3,0.327452,0.273347
4,0.231088,0.261266
5,0.193387,0.226683


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=0.46462403774261474, metrics={'train_runtime': 383.3661, 'train_samples_per_second': 2.087, 'train_steps_per_second': 0.13, 'total_flos': 8753520489984.0, 'train_loss': 0.46462403774261474, 'epoch': 5.0})

In [72]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# Model predictions
predictions = trainer.predict(tokenized_test)

# Predicted labels
preds = np.argmax(predictions.predictions, axis=-1)

# True labels
true_labels = tokenized_test["labels"]

y_true = []
y_pred = []

# -100 ko ignore karke actual word labels compare karo
for i in range(len(true_labels)):
    for j in range(len(true_labels[i])):
        if true_labels[i][j] != -100:
            y_true.append(true_labels[i][j])
            y_pred.append(preds[i][j])

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

print("Overall Accuracy:", accuracy)

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=["URD", "ENG", "MIX"],
        zero_division=0
    )
)

Overall Accuracy: 0.9209302325581395

Classification Report:
              precision    recall  f1-score   support

         URD       0.94      0.89      0.92       207
         ENG       0.90      0.94      0.92       215
         MIX       1.00      1.00      1.00         8

    accuracy                           0.92       430
   macro avg       0.95      0.95      0.95       430
weighted avg       0.92      0.92      0.92       430



In [73]:
test_sentences = [
    "Maine msging ki",
    "Tumne file ko downlod kiya",
    "Mujhe kal meetng attend karni hai",
    "Usne mera passwrd change kiya",
    "Bhai meri presentatn ready hai"
]

for sentence in test_sentences:
    print("\nSentence:", sentence)
    print("Prediction:", predict_labels(sentence))


Sentence: Maine msging ki
Prediction: [('Maine', 'URD'), ('msging', 'MIX'), ('ki', 'URD')]

Sentence: Tumne file ko downlod kiya
Prediction: [('Tumne', 'URD'), ('file', 'URD'), ('ko', 'URD'), ('downlod', 'URD'), ('kiya', 'URD')]

Sentence: Mujhe kal meetng attend karni hai
Prediction: [('Mujhe', 'URD'), ('kal', 'URD'), ('meetng', 'MIX'), ('attend', 'ENG'), ('karni', 'URD'), ('hai', 'URD')]

Sentence: Usne mera passwrd change kiya
Prediction: [('Usne', 'URD'), ('mera', 'URD'), ('passwrd', 'URD'), ('change', 'ENG'), ('kiya', 'URD')]

Sentence: Bhai meri presentatn ready hai
Prediction: [('Bhai', 'URD'), ('meri', 'URD'), ('presentatn', 'MIX'), ('ready', 'ENG'), ('hai', 'URD')]


In [74]:
save_path = "./xlm_roberta_token_classifier"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")
print("Saved at:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Saved at: ./xlm_roberta_token_classifier
